
# Algoritmo de Metropolis para o Modelo de Ising 2D

**Material de apoio ao TCC:** *Comparação e Otimização de Algoritmos de Monte Carlo
Aplicados ao Modelo de Ising* (Diego R. Oliveira, UFPR).

Este notebook implementa o algoritmo de **Metropolis** com dinâmica de inversão de
spin único (*single spin-flip*) para o modelo de Ising bidimensional, seguindo
fielmente a formulação apresentada nas Seções **2 (Modelo de Ising)**, **3.4
(Metropolis)** e **4.3.1 (Algoritmo de Metropolis)** do trabalho.

O objetivo é que este código sirva tanto para a validação e coleta de dados do TCC
quanto como **material didático para outros alunos da graduação** que queiram
entender, na prática, como uma simulação de Monte Carlo é construída do zero.

**Estrutura deste notebook:**
1. Representação da rede e estado inicial
2. Tabela de vizinhos (condições de contorno periódicas)
3. Cálculo de energia e magnetização (definições globais, usadas para validação)
4. Variação local de energia ($\Delta E$) e seus valores discretos
5. Tabela de busca (*lookup table*) para os fatores de Boltzmann
6. Critério de aceitação de Metropolis e um passo de atualização
7. Protocolo completo de simulação (equilibração + produção)
8. Testes de sanidade internos (conferência do código)
9. Validação preliminar contra a solução analítica de Onsager

Cada seção de código traz, em comentário, a equação correspondente do TCC (por
exemplo, `Eq. (4.6)`), para que o código possa ser lido lado a lado com o texto.

**Organização do código:** as funções que descrevem o *sistema físico* (rede,
vizinhos, energia, magnetização, solução de Onsager) são comuns aos três
algoritmos comparados no TCC e por isso vivem em um módulo compartilhado,
[`ising_utils.py`](https://github.com/SEU-USUARIO/ising-monte-carlo/blob/main/ising_utils.py),
importado logo abaixo. Apenas o que é *específico* do Metropolis (variação
local de energia, tabela de Boltzmann, critério de aceitação, protocolo de
simulação) é definido diretamente neste notebook.


In [ ]:

# Clona o repositório do projeto (contém o módulo compartilhado ising_utils.py)
# e adiciona ao caminho de importação do Python.

!git clone https://github.com/diegorafael1010/ising-monte-carlo.git
import sys
sys.path.append('/content/ising-monte-carlo')


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from ising_utils import (
    inicializar_rede,
    construir_tabela_vizinhos,
    energia_total,
    magnetizacao_total,
    temperatura_critica_onsager,
    energia_onsager,
    magnetizacao_onsager,
    J,
)

# Gerador de números aleatórios com semente fixa, para que os resultados deste
# notebook sejam reprodutíveis. Para rodadas de produção "de verdade", pode-se
# trocar a semente ou deixar aleatória (rng = np.random.default_rng()).
SEED = 42
rng = np.random.default_rng(SEED)



## 1. Representação da Rede e Estado Inicial

Seguindo a Seção 2.3 (Rede) e a Seção 4.3.1 do TCC, a rede quadrada de tamanho
linear $L$ é armazenada como um **arranjo unidimensional** de tamanho
$N = L^2$, com cada posição guardando o valor do spin $s_i \in \{-1, +1\}$.

Aqui usamos indexação **0-based** (padrão em Python), diferente da indexação
1-based usada na notação matemática do TCC (Equação 2.4). A relação entre a
posição de rede $(i, j)$ (linha, coluna) e o índice linear $k$ é:

$$k = i \cdot L + j, \qquad i, j \in \{0, 1, \dots, L-1\}.$$

Duas configurações iniciais são suportadas, como descrito no TCC:

- **Fria (`"fria"`)**: todos os spins iguais (estado ferromagnético ordenado,
  característico do limite $T \to 0$).
- **Quente (`"quente"`)**: cada spin sorteado independentemente (estado
  paramagnético desordenado, característico do limite $T \to \infty$).

A função `inicializar_rede` já foi importada de `ising_utils` na célula de
importações acima (é comum aos três algoritmos). O teste abaixo apenas
confirma seu comportamento.


In [ ]:

# Teste rápido: uma rede 4x4 fria deve ter todos os spins iguais a +1.
teste_fria = inicializar_rede(4, modo="fria")
assert np.all(teste_fria == 1), "Falha: rede fria deveria ter todos os spins +1"
print("Rede fria (L=4):", teste_fria)

# Uma rede quente deve, em média, ter magnetização próxima de zero para N grande.
teste_quente = inicializar_rede(64, modo="quente")
print("Magnetização média da rede quente (L=64):", teste_quente.mean(), "(esperado: perto de 0)")



## 2. Tabela de Vizinhos com Condições de Contorno Periódicas

Conforme a Seção 2.4 (Condições de Contorno) do TCC, adotam-se **condições de
contorno periódicas (PBC)**: o vizinho à direita do último sítio de uma linha é
o primeiro sítio da mesma linha, e de forma análoga para as demais direções.

Em vez de recalcular os vizinhos a cada passo (o que exigiria aritmética modular
repetida — `%`), pré-computamos uma **tabela de vizinhos**: para cada sítio $k$,
guardamos os índices lineares dos seus quatro vizinhos (norte, sul, leste,
oeste). Essa tabela é montada uma única vez, no início da simulação, e depois
apenas consultada — exatamente a otimização mencionada na Seção 2.3 do TCC
(*"tabelas de vizinhos alocadas na memória cache"*).

A função `construir_tabela_vizinhos` também já foi importada de
`ising_utils` (é idêntica para os três algoritmos, já que todos operam sobre
a mesma rede e a mesma Hamiltoniana de primeiros vizinhos).


In [ ]:

# Teste de sanidade da tabela de vizinhos: numa rede 3x3, o vizinho ao norte
# do sítio (0,0) [k=0] deve ser o sítio (2,0) [k=6], pela periodicidade.
viz_teste = construir_tabela_vizinhos(3)
assert viz_teste[0, 0] == 6, "Falha na periodicidade do vizinho norte"
print("Tabela de vizinhos para L=3, sítio k=0 (norte, sul, leste, oeste):", viz_teste[0])
print("OK: condições de contorno periódicas verificadas.")



## 3. Energia e Magnetização (Cálculo Global)

Estas funções calculam a energia total (Equação 2.6) e a magnetização total
(Equação 2.9) percorrendo **toda** a rede. Elas têm custo $\mathcal{O}(N)$ e
**não** são usadas a cada tentativa de flip (isso seria proibitivamente lento
— ver a discussão da Seção 4.3.1 sobre a vantagem de calcular $\Delta E$
localmente). Servem para:

- Definir a energia inicial da simulação;
- Validar, por conferência independente, que os cálculos incrementais de
  $\Delta E$ (Seção 4) estão corretos (Seção 8 deste notebook);
- Calcular as grandezas físicas apenas nos instantes de amostragem, durante a
  etapa de produção (bem mais raros que as tentativas de flip).

Adota-se a convenção de unidades reduzidas com $J = 1$ e $k_B = 1$, como
estabelecido na Seção 4.2 do TCC. As funções `energia_total` e
`magnetizacao_total` (assim como a constante `J`) já foram importadas de
`ising_utils`, pois são idênticas para os três algoritmos.


In [ ]:

# Teste de sanidade: numa rede 2x2 totalmente alinhada (fria), cada spin tem
# 4 vizinhos, mas em L=2 com PBC cada vizinho North/South/East/West aponta
# para os mesmos 2 outros sítios -- então usamos L=4 para um teste mais claro.
L_teste = 4
estado_teste = inicializar_rede(L_teste, modo="fria")
viz_teste = construir_tabela_vizinhos(L_teste)

E_teste = energia_total(estado_teste, viz_teste)
M_teste = magnetizacao_total(estado_teste)

# Para uma rede totalmente alinhada, todas as N*2 ligações contribuem com -J.
N_teste = L_teste ** 2
E_esperada = -J * 2 * N_teste  # 2 ligações "novas" por sítio (direita e baixo)
print(f"Energia total (rede fria, L={L_teste}): {E_teste} (esperado: {E_esperada})")
print(f"Magnetização total (rede fria, L={L_teste}): {M_teste} (esperado: {N_teste})")
assert E_teste == E_esperada
assert M_teste == N_teste
print("OK: energia e magnetização globais conferem com o valor esperado analiticamente.")



## 4. Variação Local de Energia ($\Delta E$)

Como discutido na Seção 4.3.1 (e na Seção 2.5) do TCC, a variação de energia
associada à tentativa de inversão de um único spin $s_k$ depende **apenas**
dos seus quatro primeiros vizinhos, e é dada pela Equação (4.6):

$$\Delta E_k = 2 J s_k \sum_{j \in nn(k)} s_j$$

Como cada $s_j \in \{-1, +1\}$ e há exatamente 4 vizinhos, a soma
$\sum_{j\in nn(k)} s_j$ só pode assumir os valores $\{-4, -2, 0, +2, +4\}$
(Equação 4.7), de modo que, com $J=1$, $\Delta E_k$ só pode assumir os cinco
valores discretos $\{-8, -4, 0, +4, +8\}$ (Equação 4.8). Isso é o que torna
possível pré-calcular os fatores de Boltzmann na Seção 5 a seguir.


In [ ]:

def delta_E_local(estado: np.ndarray, vizinhos: np.ndarray, k: int) -> float:
    '''Calcula a variação de energia associada à inversão do spin k
    (Equação 4.6 do TCC), sem recalcular a energia de toda a rede.
    '''
    soma_vizinhos = estado[vizinhos[k]].sum()
    return 2.0 * J * estado[k] * soma_vizinhos


In [ ]:

# Teste de sanidade: o conjunto de valores possíveis de delta_E_local deve
# ser exatamente {-8, -4, 0, 4, 8}, para qualquer configuração (Equação 4.8).
L_teste = 16
estado_teste = inicializar_rede(L_teste, modo="quente")
viz_teste = construir_tabela_vizinhos(L_teste)

valores_encontrados = set()
for k in range(L_teste ** 2):
    valores_encontrados.add(delta_E_local(estado_teste, viz_teste, k))

print("Valores de delta_E encontrados:", sorted(valores_encontrados))
assert valores_encontrados.issubset({-8.0, -4.0, 0.0, 4.0, 8.0})
print("OK: delta_E_local só assume os cinco valores previstos pela Equação 4.8.")



## 5. Tabela de Busca (*Lookup Table*) para os Fatores de Boltzmann

Como $\Delta E \le 0$ implica aceitação garantida ($P_{acc}=1$), só é
necessário avaliar o fator de Boltzmann $e^{-\beta \Delta E}$ para os dois
valores positivos possíveis, $\Delta E \in \{+4, +8\}$ (Equação 4.9 do TCC).
Pré-calculamos esses dois valores **uma única vez** por temperatura, no início
da simulação, evitando avaliar exponenciais repetidamente dentro do laço
principal do Metropolis.


In [ ]:

def construir_lookup_boltzmann(beta: float) -> dict:
    '''Pré-calcula os fatores de Boltzmann para as duas variações de energia
    positivas possíveis (Equação 4.9 do TCC).

    Parameters
    ----------
    beta : float
        Temperatura inversa, beta = 1/T (unidades reduzidas, k_B = 1).

    Returns
    -------
    dict
        Dicionário {4.0: exp(-beta*4), 8.0: exp(-beta*8)}.
    '''
    return {dE: np.exp(-beta * dE) for dE in (4.0, 8.0)}


In [ ]:

# Teste de sanidade: para beta muito grande (T -> 0), os fatores de Boltzmann
# devem ser próximos de zero (aceitar um aumento de energia fica muito raro).
lookup_frio = construir_lookup_boltzmann(beta=10.0)
print("Lookup table para T baixa (beta=10):", lookup_frio)
assert lookup_frio[8.0] < lookup_frio[4.0] < 0.1
print("OK: fatores de Boltzmann decrescem com o aumento de delta_E, como esperado.")



## 6. Critério de Aceitação de Metropolis e um Passo de Atualização

O critério de aceitação (Equação 3.14 do TCC) é:

$$A(\mu \to \nu) = \min\left(1, e^{-\beta \Delta E}\right)$$

ou seja: se $\Delta E \le 0$, a inversão é aceita imediatamente; caso
contrário, é aceita com probabilidade $e^{-\beta \Delta E}$ (consultada na
*lookup table* construída acima).

Um **passo de Monte Carlo por sítio (1 MCS/site)** corresponde a $N = L^2$
tentativas de inversão (Seção 4.3.1). Implementamos os dois esquemas de
seleção de sítio discutidos no TCC:

- `"aleatoria"`: o sítio é sorteado uniformemente a cada tentativa;
- `"sequencial"`: a rede é varrida em ordem, do sítio 0 ao $N-1$.

Ambos os esquemas produzem, no equilíbrio, as mesmas médias estatísticas
(Seção 4.3.1), mas a seleção sequencial tem melhor aproveitamento de cache.


In [ ]:

def tentativa_metropolis(
    estado: np.ndarray,
    vizinhos: np.ndarray,
    k: int,
    lookup: dict,
    rng: np.random.Generator,
) -> bool:
    '''Executa uma única tentativa de inversão do spin k, seguindo o
    critério de Metropolis (Equação 3.14). Modifica `estado` in-place se a
    tentativa for aceita.

    Returns
    -------
    bool
        True se a inversão foi aceita, False caso contrário.
    '''
    dE = delta_E_local(estado, vizinhos, k)
    if dE <= 0:
        aceito = True
    else:
        r = rng.random()
        aceito = r < lookup[dE]

    if aceito:
        estado[k] *= -1

    return aceito


def sweep_metropolis(
    estado: np.ndarray,
    vizinhos: np.ndarray,
    lookup: dict,
    rng: np.random.Generator,
    esquema: str = "aleatoria",
) -> int:
    '''Executa 1 MCS/site: N = L**2 tentativas de inversão (Seção 4.3.1).

    Parameters
    ----------
    esquema : {"aleatoria", "sequencial"}
        Esquema de escolha do sítio a cada tentativa.

    Returns
    -------
    int
        Número de inversões aceitas nesse sweep (útil como diagnóstico da
        taxa de aceitação, mas não é uma grandeza física do TCC).
    '''
    N = estado.shape[0]
    if esquema == "aleatoria":
        sitios = rng.integers(0, N, size=N)
    elif esquema == "sequencial":
        sitios = np.arange(N)
    else:
        raise ValueError("esquema deve ser 'aleatoria' ou 'sequencial'")

    aceitos = 0
    for k in sitios:
        if tentativa_metropolis(estado, vizinhos, int(k), lookup, rng):
            aceitos += 1
    return aceitos


In [ ]:

# Teste de sanidade: a beta = 0 (T infinita), TODA tentativa deve ser aceita,
# já que exp(-0*dE) = 1 sempre.
L_teste = 8
estado_teste = inicializar_rede(L_teste, modo="quente")
viz_teste = construir_tabela_vizinhos(L_teste)
lookup_infinito = construir_lookup_boltzmann(beta=0.0)

aceitos = sweep_metropolis(estado_teste, viz_teste, lookup_infinito, rng, esquema="aleatoria")
print(f"Aceitos em T=infinito: {aceitos} de {L_teste**2} tentativas (esperado: {L_teste**2})")
assert aceitos == L_teste ** 2
print("OK: a T infinita, todas as tentativas são aceitas, como esperado.")



## 7. Protocolo Completo de Simulação (Equilibração + Produção)

Reunindo as peças anteriores, esta função executa o protocolo descrito na
Seção 4.5.3 do TCC:

1. Inicializa a rede (fria ou quente);
2. Executa `n_eq` MCS/site de **equilíbração**, descartando todas as
   configurações geradas;
3. Executa `n_prod` MCS/site de **produção**, registrando energia e
   magnetização a cada `intervalo_amostragem` MCS/site.

Os valores de produção usados no TCC (Equações 4.25 e 4.26: $N_{eq}=10^5$,
$N_{prod}=10^6$) são grandes para fins de demonstração rápida neste notebook;
aqui usamos valores bem menores nos testes, e o valor final de produção só
deve ser usado nas rodadas oficiais de coleta de dados (mais lentas).


In [ ]:

def simular_metropolis(
    L: int,
    T: float,
    n_eq: int,
    n_prod: int,
    intervalo_amostragem: int = 10,
    modo_inicial: str = "quente",
    esquema: str = "aleatoria",
    rng: np.random.Generator = rng,
) -> dict:
    '''Executa o protocolo completo de simulação de Metropolis para o
    modelo de Ising 2D (Seções 4.3.1 e 4.5.3 do TCC).

    Parameters
    ----------
    L : int
        Tamanho linear da rede.
    T : float
        Temperatura reduzida (k_B = 1).
    n_eq : int
        Número de MCS/site de equilibração (descartados).
    n_prod : int
        Número de MCS/site de produção (amostrados).
    intervalo_amostragem : int
        A cada quantos MCS/site uma medida é registrada.
    modo_inicial : {"fria", "quente"}
        Configuração inicial da rede.
    esquema : {"aleatoria", "sequencial"}
        Esquema de seleção de sítios em cada sweep.

    Returns
    -------
    dict
        Dicionário com:
        - "energia_por_sitio": array com e(t) = E(t)/N amostrado
        - "magnetizacao_abs_por_sitio": array com |m(t)| = |M(t)|/N amostrado
        - "L", "T", "n_eq", "n_prod": parâmetros usados (para registro)
    '''
    N = L * L
    beta = 1.0 / T

    estado = inicializar_rede(L, modo=modo_inicial, rng=rng)
    vizinhos = construir_tabela_vizinhos(L)
    lookup = construir_lookup_boltzmann(beta)

    # --- Equilibração: descarta todas as configurações geradas ---
    for _ in range(n_eq):
        sweep_metropolis(estado, vizinhos, lookup, rng, esquema=esquema)

    # --- Produção: amostra a cada `intervalo_amostragem` MCS/site ---
    energias = []
    magnetizacoes = []
    for passo in range(n_prod):
        sweep_metropolis(estado, vizinhos, lookup, rng, esquema=esquema)
        if passo % intervalo_amostragem == 0:
            e = energia_total(estado, vizinhos) / N
            m = abs(magnetizacao_total(estado)) / N
            energias.append(e)
            magnetizacoes.append(m)

    return {
        "energia_por_sitio": np.array(energias),
        "magnetizacao_abs_por_sitio": np.array(magnetizacoes),
        "L": L,
        "T": T,
        "n_eq": n_eq,
        "n_prod": n_prod,
    }


In [ ]:

# Teste rápido (parâmetros pequenos só para conferir que a função roda e que
# a energia parece razoável -- NÃO é uma rodada de produção real).
resultado_teste = simular_metropolis(L=16, T=2.5, n_eq=200, n_prod=1000, intervalo_amostragem=5)
print("Energia média por sítio:", resultado_teste["energia_por_sitio"].mean())
print("Magnetização absoluta média por sítio:", resultado_teste["magnetizacao_abs_por_sitio"].mean())

plt.figure(figsize=(7, 3))
plt.plot(resultado_teste["energia_por_sitio"])
plt.xlabel("Amostra (a cada 5 MCS/site)")
plt.ylabel(r"Energia por sítio $e$")
plt.title(f"Traço de energia -- L={resultado_teste['L']}, T={resultado_teste['T']}")
plt.tight_layout()
plt.show()



## 8. Testes de Sanidade Internos

Antes de confiar no código para gerar dados do TCC, vale a pena conferir que
o cálculo **incremental** de $\Delta E$ (usado a cada tentativa, por ser
rápido) concorda com o cálculo **direto**, feito recalculando a energia total
antes e depois da inversão (mais lento, mas inequivocamente correto por
definição).

Isso é diferente dos testes de sanidade da Seção 4.4.3 do TCC (que
introduzem erros de propósito para verificar a sensibilidade do código a
vieses de implementação); aqui verificamos apenas a **corretude aritmética**
do cálculo incremental.


In [ ]:

def teste_consistencia_delta_E(L: int = 10, n_testes: int = 200, rng: np.random.Generator = rng) -> None:
    '''Confere, para várias configurações e sítios aleatórios, que o
    delta_E incremental bate com a diferença de energia total calculada
    diretamente (antes e depois da inversão).
    '''
    vizinhos = construir_tabela_vizinhos(L)

    for _ in range(n_testes):
        estado = inicializar_rede(L, modo="quente", rng=rng)
        k = int(rng.integers(0, L * L))

        E_antes = energia_total(estado, vizinhos)
        dE_incremental = delta_E_local(estado, vizinhos, k)

        estado[k] *= -1  # aplica a inversão manualmente
        E_depois = energia_total(estado, vizinhos)
        dE_direto = E_depois - E_antes

        assert np.isclose(dE_incremental, dE_direto), (
            f"Inconsistência: incremental={dE_incremental}, direto={dE_direto}"
        )

    print(f"OK: delta_E incremental consistente com o cálculo direto em {n_testes} testes.")

teste_consistencia_delta_E()



## 9. Validação Preliminar contra a Solução Analítica de Onsager

Esta seção antecipa, de forma simplificada e rápida, o teste completo descrito
na Seção 4.4.1 do TCC: comparar a energia e a magnetização médias simuladas
com as curvas exatas de Onsager (Equações 4.21 e 4.22).

**Atenção:** esta é apenas uma checagem rápida de sanidade, com poucos MCS de
produção e uma única rede pequena — **não** substitui a validação completa e
estatisticamente rigorosa da Seção 4.4 do TCC, que deve ser feita com os
valores de produção definidos na Seção 4.5.3 ($N_{eq}=10^5$,
$N_{prod}=10^6$) e várias sementes/repetições.

Nota sobre a convenção do SciPy: a função `scipy.special.ellipk(m)` espera o
**parâmetro** $m = k_1^2$, não o módulo $k_1$ diretamente (Equação 4.23 do
TCC usa o módulo $k_1$) -- isso já está tratado dentro de `energia_onsager`,
em `ising_utils`. As três funções usadas abaixo
(`temperatura_critica_onsager`, `energia_onsager`, `magnetizacao_onsager`)
já foram importadas de lá, pois servirão também para validar Wolff e
Swendsen-Wang nos respectivos notebooks.


In [ ]:

# ATENÇÃO: valores pequenos só para demonstração -- rodar em poucos segundos.
# Para a validação oficial do TCC, usar os parâmetros da Seção 4.5 (L maior,
# N_eq=1e5, N_prod=1e6) e comparar com barras de erro (método da Seção 4.7).

L_demo = 32
temperaturas_demo = np.array([1.5, 2.0, 2.269, 2.5, 3.0])

e_simulado = []
m_simulado = []
for T in temperaturas_demo:
    r = simular_metropolis(L=L_demo, T=T, n_eq=2000, n_prod=5000, intervalo_amostragem=5)
    e_simulado.append(r["energia_por_sitio"].mean())
    m_simulado.append(r["magnetizacao_abs_por_sitio"].mean())

e_simulado = np.array(e_simulado)
m_simulado = np.array(m_simulado)

temperaturas_finas = np.linspace(1.2, 3.5, 200)
e_exato = np.array([energia_onsager(T) for T in temperaturas_finas])
m_exato = np.array([magnetizacao_onsager(T) for T in temperaturas_finas])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(temperaturas_finas, e_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax1.plot(temperaturas_demo, e_simulado, "o", color="crimson", label=f"Metropolis (L={L_demo}, demo rápida)")
ax1.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax1.set_xlabel("Temperatura $T$")
ax1.set_ylabel("Energia por sítio $e$")
ax1.legend()

ax2.plot(temperaturas_finas, m_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax2.plot(temperaturas_demo, m_simulado, "o", color="crimson", label=f"Metropolis (L={L_demo}, demo rápida)")
ax2.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax2.set_xlabel("Temperatura $T$")
ax2.set_ylabel("Magnetização absoluta por sítio $|m|$")
ax2.legend()

plt.suptitle("Validação preliminar (demonstração rápida -- não é a validação oficial do TCC)")
plt.tight_layout()
plt.show()



## Próximos Passos

- **Validação oficial (Seção 4.4 do TCC):** repetir a comparação acima com os
  parâmetros reais de produção ($N_{eq}=10^5$, $N_{prod}=10^6$, redes
  $L \in \{16, 32, 64, 128\}$) e reportar incertezas estatísticas (ver
  notebook de autocorrelação/blocking, Seção 4.7).
- **Testes de sanidade da Seção 4.4.3:** implementar as duas contra-provas
  (omitir a recontagem de rejeições; quebrar o balanço detalhado) em um
  notebook separado, e confirmar que ambas produzem os desvios esperados em
  relação à curva de Onsager.
- **Wolff e Swendsen-Wang:** implementar em notebooks próprios
  (`wolff.ipynb`, `swendsen_wang.ipynb`), importando as funções comuns do
  mesmo módulo compartilhado `ising_utils.py` usado aqui (`inicializar_rede`,
  `construir_tabela_vizinhos`, `energia_total`, `magnetizacao_total`,
  `probabilidade_ligacao`), da mesma forma que este notebook faz.
- **Nota sobre desempenho:** esta implementação prioriza clareza didática,
  não velocidade máxima. A otimização do código (por exemplo, compilação
  *just-in-time* com Numba, ou paralelização) está fora do escopo desta
  primeira etapa do trabalho (ver Seção 4.9 do TCC) e será tratada,
  quando pertinente, na etapa seguinte da pesquisa (Introdução à Pesquisa
  II).
